# Brain Tumor Classification (MRI)

PyTorch + timm + optional MONAI

## 2. Project Overview

Build a brain tumor image classifier with transfer learning on MRI slices and clearly document the limitations of slice-level classification.

## 3. Learning Objectives

- Train a transfer-learning MRI classifier
- Use timm backbones with medical-image-safe preprocessing
- Evaluate with clinically relevant class-wise metrics
- Understand why slice-level classification is limited

## 4. Problem Statement

Given an MRI slice, predict tumor class or no-tumor class depending on the dataset labeling scheme.

## 5. Why This Project Matters

Automated MRI triage can support radiology workflows, but benchmark success is not equivalent to clinical readiness.

## 6. Dataset Overview

This notebook targets a brain tumor MRI image dataset commonly organized into class folders. The exact class set may vary by source (for example: glioma, meningioma, pituitary, no_tumor).

## 7. Dataset Source and License Notes

Use the dataset only according to the original source license and any medical-data governance requirements.

## 8. Environment Setup

Required: torch, timm, albumentations, scikit-learn, matplotlib, seaborn. Optional: MONAI for medical-image-specific workflows.

In [ ]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import cv2
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix, classification_report

warnings.filterwarnings('ignore')

try:
    import monai
    HAS_MONAI = True
except Exception:
    HAS_MONAI = False

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('timm:', timm.__version__)
print('Albumentations:', A.__version__)
print('MONAI available:', HAS_MONAI)

In [ ]:
# 10. Configuration / constants
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE = 224
BATCH_SIZE = 32
NUM_WORKERS = 0
LR = 1e-4
EPOCHS_BASELINE = 1
EPOCHS_STRONG = 1

BASELINE_MODEL = 'resnet18'
STRONG_MODEL = 'convnext_tiny'

SAVE_DIR = Path.cwd() / 'Computer Vision' / 'Brain Tumor Classification'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR = SAVE_DIR / 'brain_tumor_mri_data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Save dir:', SAVE_DIR)

In [ ]:
# 11. Dataset loading
# Default synthetic mode for deterministic execution in this environment.
FORCE_SYNTHETIC = True
use_synthetic = FORCE_SYNTHETIC

class_names = ['glioma', 'meningioma', 'pituitary', 'no_tumor']
num_classes = len(class_names)

train_images, train_labels = [], []
val_images, val_labels = [], []
test_images, test_labels = [], []

if not use_synthetic:
    # Real-data expectation:
    # DATA_DIR/<split>/<class_name>/*.jpg or similar
    pass

if use_synthetic:
    # Simulate realistic imbalance and class count
    train_counts = [800, 620, 540, 980]
    val_counts = [120, 90, 80, 150]
    test_counts = [180, 130, 110, 220]

    for cls, n in enumerate(train_counts):
        for _ in range(n):
            train_labels.append(cls)
            train_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

    for cls, n in enumerate(val_counts):
        for _ in range(n):
            val_labels.append(cls)
            val_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

    for cls, n in enumerate(test_counts):
        for _ in range(n):
            test_labels.append(cls)
            test_images.append(np.random.randint(0, 255, size=(IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8))

print('Using synthetic mode:', use_synthetic)
print('Classes:', class_names)
print('Train/Val/Test sizes:', len(train_labels), len(val_labels), len(test_labels))

In [ ]:
# 12. Data validation checks
assert len(train_images) == len(train_labels)
assert len(val_images) == len(val_labels)
assert len(test_images) == len(test_labels)
assert set(train_labels).issubset(set(range(num_classes)))

print('Validation checks passed.')
print('Train class counts:', dict(zip(class_names, np.bincount(np.array(train_labels), minlength=num_classes).tolist())))

## 13. Exploratory Data Analysis

Inspect class balance and sample slices.

In [ ]:
counts = np.bincount(np.array(train_labels), minlength=num_classes)

plt.figure(figsize=(8,4))
plt.bar(class_names, counts)
plt.xticks(rotation=20)
plt.title('Train Class Distribution')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    idx = np.where(np.array(train_labels) == i)[0][0]
    ax.imshow(train_images[idx], cmap='gray')
    ax.set_title(class_names[i])
    ax.axis('off')
plt.tight_layout()
plt.show()

## 14. Train/Validation/Test Split Strategy

Use patient-level splitting when metadata exists. Multiple slices from the same subject must not be spread across train/validation/test.

In [ ]:
# 15. Preprocessing and augmentation strategy
train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, p=0.3),
    A.RandomBrightnessContrast(0.12, 0.12, p=0.25),
    A.GaussNoise(var_limit=(3, 12), p=0.15),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)),
    ToTensorV2(),
])

print('Albumentations pipelines configured.')
if HAS_MONAI:
    print('MONAI is available for optional medical image enhancements.')

## 16. Baseline Approach

Baseline model: ResNet-18 transfer learning.

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, images, labels, transform):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        img = self.images[idx]
        y = int(self.labels[idx])
        x = self.transform(image=img)['image']
        return x, y

train_ds = BrainTumorDataset(train_images, train_labels, train_tf)
val_ds = BrainTumorDataset(val_images, val_labels, val_tf)
test_ds = BrainTumorDataset(test_images, test_labels, val_tf)

# Mild imbalance handling
class_counts = np.bincount(np.array(train_labels), minlength=num_classes)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = np.array([class_weights[y] for y in train_labels], dtype=np.float32)
sampler = WeightedRandomSampler(weights=torch.from_numpy(sample_weights), num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

baseline_model = timm.create_model(BASELINE_MODEL, pretrained=True, num_classes=num_classes).to(DEVICE)
strong_model = timm.create_model(STRONG_MODEL, pretrained=True, num_classes=num_classes).to(DEVICE)

print('Baseline model:', BASELINE_MODEL)
print('Strong model:', STRONG_MODEL)
print('Weighted sampler enabled.')

## 17. Main Model/Workflow

Stronger model: ConvNeXt-Tiny transfer learning for richer MRI texture representation.

In [ ]:
# 18. Training loop / fine-tuning pipeline
def run_epoch(model, loader, optimizer=None):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    weights = torch.tensor(1.0 / np.maximum(np.bincount(np.array(train_labels), minlength=num_classes), 1), dtype=torch.float32, device=DEVICE)
    criterion = nn.CrossEntropyLoss(weight=weights)

    total_loss = 0.0
    ys, ps = [], []

    for xb, yb in loader:
        xb = xb.to(DEVICE)
        yb = yb.to(DEVICE)

        if train_mode:
            optimizer.zero_grad()

        with torch.set_grad_enabled(train_mode):
            logits = model(xb)
            loss = criterion(logits, yb)
            if train_mode:
                loss.backward()
                optimizer.step()

        total_loss += float(loss.item())
        pred = logits.argmax(dim=1)
        ys.extend(yb.cpu().numpy().tolist())
        ps.extend(pred.cpu().numpy().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    acc = accuracy_score(ys, ps)
    macro_f1 = f1_score(ys, ps, average='macro', zero_division=0)
    macro_recall = recall_score(ys, ps, average='macro', zero_division=0)
    return avg_loss, acc, macro_f1, macro_recall, ys, ps


def train_model(model, name, epochs):
    optimizer = optim.AdamW(model.parameters(), lr=LR)
    best_f1 = -1.0
    best_state = None

    for ep in range(1, epochs + 1):
        tr_loss, tr_acc, tr_f1, tr_rec, _, _ = run_epoch(model, train_loader, optimizer=optimizer)
        va_loss, va_acc, va_f1, va_rec, _, _ = run_epoch(model, val_loader, optimizer=None)
        print(f'[{name}] ep={ep} train_acc={tr_acc:.4f} train_macro_f1={tr_f1:.4f} val_acc={va_acc:.4f} val_macro_f1={va_f1:.4f}')
        if va_f1 > best_f1:
            best_f1 = va_f1
            best_state = {k: v.cpu() for k, v in model.state_dict().items()}

    if best_state is not None:
        model.load_state_dict(best_state)

train_model(baseline_model, 'baseline', EPOCHS_BASELINE)
train_model(strong_model, 'strong', EPOCHS_STRONG)

## 19. Inference Examples

Deployment-style top-k response for MRI slice review.

In [ ]:
def infer_single(model, image_rgb, k=3):
    model.eval()
    x = val_tf(image=image_rgb)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    top_idx = np.argsort(-probs)[:k]
    preds = []
    for i in top_idx:
        preds.append({
            'class_id': int(i),
            'class_name': class_names[int(i)],
            'confidence': float(probs[int(i)])
        })

    return {
        'top_k': preds,
        'predicted_class': preds[0]['class_name'],
        'confidence': preds[0]['confidence']
    }

sample = test_images[0]
resp = infer_single(strong_model, sample, k=3)
print(json.dumps(resp, indent=2))

## 20. Evaluation

Report accuracy, macro-F1, macro-recall, and class-wise recall.

In [ ]:
_, b_acc, b_f1, b_rec, by, bp = run_epoch(baseline_model, test_loader, optimizer=None)
_, s_acc, s_f1, s_rec, sy, sp = run_epoch(strong_model, test_loader, optimizer=None)

print('Baseline -> acc:', round(b_acc,4), 'macro_f1:', round(b_f1,4), 'macro_recall:', round(b_rec,4))
print('Strong   -> acc:', round(s_acc,4), 'macro_f1:', round(s_f1,4), 'macro_recall:', round(s_rec,4))

cls_recall = recall_score(sy, sp, average=None, labels=list(range(num_classes)), zero_division=0)
print('Class-wise recall:')
for i, r in enumerate(cls_recall):
    print(class_names[i], round(float(r), 4))

print()
print('Classification report (strong):')
print(classification_report(sy, sp, target_names=class_names, zero_division=0))

## 21. Confusion Matrix and Error Analysis

In [ ]:
cm = confusion_matrix(sy, sp, labels=list(range(num_classes)))
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(8, 6))
sns.heatmap(cmn, cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xticks(rotation=20)
plt.yticks(rotation=0)
plt.title('Normalized Confusion Matrix (Strong Model)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

## 22. Slice-Level Limitations Discussion

Important limitation: this notebook classifies **individual slices**, not full MRI volumes. That creates real constraints:

1. **Lost 3D context**: tumor extent and structure across adjacent slices are ignored
2. **Slice selection bias**: one slice may miss the most informative tumor region
3. **Patient-level leakage risk**: nearby slices from the same patient must not cross splits
4. **Clinical mismatch**: radiologists interpret sequences and full studies, not isolated JPEG slices
5. **Weak volume-level claims**: strong slice accuracy does not imply strong patient-level diagnosis

For real deployment, prefer patient-level and volume-aware evaluation.

## 23. Medical Caveats

This is not a diagnostic tool.

Key caveats:
1. Dataset composition may not reflect real hospital populations
2. MRI protocols/devices vary across institutions
3. Ground truth may differ by source and labeling protocol
4. Model output must be interpreted with radiologist oversight
5. Benchmark performance does not equal clinical approval

## 24. How To Improve

- Move from slice-level to volume-level modeling
- Use MONAI pipelines for medical-image workflows
- Add external validation and calibration
- Incorporate sequence metadata and multi-slice context

## 25. Production Considerations

- Enforce patient-level split policy and audit it
- Track class-wise recall drift
- Add human review for uncertain or high-risk cases

## 26. Common Mistakes

- Treating slice classification as patient diagnosis
- Random image-level split with patient leakage
- Reporting only accuracy without class-wise recall

## 27. Mini Challenge / Final Summary

Mini challenge: aggregate predictions across slices for a simulated patient-level classifier.

Summary: this notebook builds a transfer-learning MRI classifier and clearly explains why slice-level evaluation is limited.

In [ ]:
# Save metrics
train_counts = np.bincount(np.array(train_labels), minlength=num_classes)

metrics = {
    'dataset': 'Brain Tumor MRI dataset',
    'num_classes': int(num_classes),
    'baseline_model': BASELINE_MODEL,
    'strong_model': STRONG_MODEL,
    'baseline_test_accuracy': float(b_acc),
    'baseline_test_macro_f1': float(b_f1),
    'baseline_test_macro_recall': float(b_rec),
    'strong_test_accuracy': float(s_acc),
    'strong_test_macro_f1': float(s_f1),
    'strong_test_macro_recall': float(s_rec),
    'class_wise_recall_strong': {class_names[i]: float(cls_recall[i]) for i in range(num_classes)},
    'imbalance_ratio_train_max_min': float((train_counts.max()+1)/(train_counts.min()+1)),
    'device': str(DEVICE),
    'synthetic_mode': bool(use_synthetic),
    'monai_available': bool(HAS_MONAI)
}

metrics_path = SAVE_DIR / 'metrics.json'
with open(metrics_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('Saved metrics:', metrics_path)
print('Done.')